In [2]:
SYSTEM_PROMPT = """
You are a data generation assistant for an Indonesian UMKM CRM classifier.
Generate realistic customer DM messages in Bahasa Indonesia (informal/colloquial).
A Transitional customer: has low-to-medium commitment, is still comparing options,
asking follow-up questions, or warming up after a prior interaction.
They are NOT clearly loyal (Communal) and NOT just complaining (Transactional).

Output ONLY a JSON array. No preamble, no markdown fences.
"""

ARCHETYPES = {
    "price_comparison": """
Generate {n} Transitional customer DMs where the customer is asking about price
and explicitly or implicitly comparing with another seller.
Example tone: "Kak harga X berapa ya? Soalnya tadi liat tempat lain agak beda..."
Output: [{{"content": "...", "label": 1}}]
""",
    "repeat_enquirer": """
Generate {n} Transitional customer DMs where the customer has bought before (1-2x)
and is asking about a new or related product. Shows some familiarity but not committed.
Example tone: "Kemarin beli di sini enak kak, kalau yang varian B ada ga?"
Output: [{{"content": "...", "label": 1}}]
""",
    "conditional_buyer": """
Generate {n} Transitional customer DMs where the customer wants to buy
but adds a condition (promo, ready stock, minimum order, delivery area).
Example tone: "Kalau beli 3 bisa dikurangin ongkirnya ga kak?"
Output: [{{"content": "...", "label": 1}}]
""",
    "slow_negotiation": """
Generate {n} Transitional multi-turn DM snippets (2-3 messages from customer side only)
showing a customer slowly negotiating price or terms, not yet committing.
Example tone: "Oke, terus kalau bayar dp dulu bisa? ... Ntar saya konfirm ya kak"
Output: [{{"content": "...", "label": 1}}]
""",
    "dormant_reactivation": """
Generate {n} Transitional customer DMs from a customer who hasn't bought in a while
and is just checking back in or asking if a product is still available.
Example tone: "Masih jual yang dulu itu ga kak? Udah lama ga order"
Output: [{{"content": "...", "label": 1}}]
"""
}

In [ ]:
from google import genai
import os

In [9]:
model="gemini-2.5-flash-lite"

In [ ]:
client = genai.Client(api_key="API_KEY")
response = client.models.generate_content(
    model=model, 
    contents="Say hello"
)
print(response.text)

Hello!


In [5]:
BATCH_SIZE = 50 
ARCHETYPE_TARGETS = {
    "price_comparison":    1500,
    "repeat_enquirer":     1200,
    "conditional_buyer":   1000,
    "slow_negotiation":     800,
    "dormant_reactivation": 500,
}

In [6]:
import json
import time

In [ ]:
# TEST
prompt_template = ARCHETYPES["price_comparison"]
user_prompt = prompt_template.format(n=1)
response = client.models.generate_content(
  model=model,
  config=genai.types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    response_mime_type="application/json",
    temperature=0.9
  ),
  contents=user_prompt
)

raw = response.text
batch = json.loads(raw)


[
  {"content": "Kak, mau tanya nih, untuk produk A ini harganya berapa ya? Tadi di toko sebelah kok kayaknya lebih murah deh, apa ada perbedaan kualitas?", "label": 1}
]


In [ ]:
from google.api_core import exceptions as google_exceptions
import sys

In [ ]:
all_samples = []
MAX_RETRIES = 5

for archetype, total in ARCHETYPE_TARGETS.items():
    collected = []
    prompt_template = ARCHETYPES[archetype]

    while len(collected) < total:
        n = min(BATCH_SIZE, total - len(collected))
        user_prompt = prompt_template.format(n=n)
        retries = 0

        while retries < MAX_RETRIES:
            try:
                response = client.models.generate_content(
                    model=model,
                    config=genai.types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        response_mime_type="application/json",
                        temperature=0.9
                    ),
                    contents=user_prompt
                )
                break  # success

            except google_exceptions.ResourceExhausted:
                retries += 1
                wait = 7.5 * retries  # incremental backoff
                print(f"# 429 - rate limited, retry {retries}/{MAX_RETRIES} in {wait}s")
                time.sleep(wait)

            except google_exceptions.ServiceUnavailable:
                retries += 1
                print(f"# 503 - model overloaded, retry {retries}/{MAX_RETRIES}")
                time.sleep(7.5)

        else:
            print(f"# Max retries hit for {archetype}, aborting.")
            sys.exit(1)

        try:
            batch = json.loads(response.text)
        except json.JSONDecodeError as e:
            print(f"# JSON parse error: {e} — skipping batch")
            time.sleep(7.5)
            continue

        if isinstance(batch, list):
            samples = batch
        else:
            samples = next((v for v in batch.values() if isinstance(v, list)), [])

        valid = [
            s for s in samples
            if isinstance(s.get("content"), str)
            and len(s["content"]) > 15
            and s.get("label") == 1
        ]

        collected.extend(valid)
        print(f"{archetype}: {len(collected)}/{total}")
        time.sleep(7.5)

    all_samples.extend(collected[:total])

In [ ]:
# save
import pandas as pd
df_new = pd.DataFrame(all_samples)
df_new.to_csv("transitional_synthetic_5000.csv", index=False)
print(f"Total generated: {len(df_new)}")